# Checkpoint 1 — Framing the Problem

Business Problem:

The goal of this project is to build a machine learning model that can predict the selling price of residential properties in Azerbaijan.

House prices can vary significantly depending on factors such as location, area, number of rooms, floor, property condition, and other characteristics. A price prediction model could help real estate businesses and property owners estimate a reasonable price for a property and support faster, more informed decisions.

Target Variable:

The target variable is **`price`**, which represents the sale price of the property in **AZN (Azerbaijani Manat)**.

Before starting the modelling process, I checked that:

* the `price` column exists;
* it is numeric;
* it contains no missing values;
* all prices are positive.

These checks confirmed that `price` is suitable as the prediction target.

Evaluation Metrics:

The main metric for this project is **Mean Absolute Error (MAE)** because it is easy to interpret in business terms. For example, an MAE of 100,000 AZN means that the model's predictions are off by about 100,000 AZN on average.

I will also use **Root Mean Squared Error (RMSE)** and **R²** to get a more complete view of model performance.

The main goal is to **minimize MAE**, while RMSE and R² will be used as supporting metrics when comparing the models.


In [122]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)

# load dataset
df = pd.read_csv("house_sale.csv")

# size of the dataset?
print("Shape:", df.shape)

# check currencies
print("\nCurrency distribution:")
print(df["currency_x"].value_counts(dropna=False))

# check the price range
print("\nPrice range:")
print(df["price"].min(), "-", df["price"].max())

# target column exists?
print("\nTarget column:", "price")

# check the number of rows
print("Rows:", len(df))

# check if there are missing prices
print("Missing prices:", df["price"].isnull().sum())

# check if there are any neagtive prices
print("Prices <= 0:", (df["price"] <= 0).sum())

# check the target data type
print("Price data type:", df["price"].dtype)

Shape: (100775, 51)

Currency distribution:
currency_x
AZN    100775
Name: count, dtype: int64

Price range:
11.0 - 600000000.0

Target column: price
Rows: 100775
Missing prices: 0
Prices <= 0: 0
Price data type: float64


In [123]:
# here the target has no missing or non-positive values

# Checkpoint 2 — Full EDA and Cleaning

In this step, I explored the dataset to understand its structure and find problems that could affect the model.

I checked the dataset size, columns, data types, missing values, duplicates, and the distribution of house prices.

I also checked for repeated property listings, unusual prices, redundant columns, and missing categorical values.

For cleaning, I removed unnecessary or duplicated columns, handled features with too many missing values, converted the area information into numerical features, and filled missing categorical values with `"Missing"`.

I also kept repeated properties together for later model validation so that the same property would not appear in both training and validation data.

After cleaning, I created the final dataset that will be used for model comparison.


In [124]:
# 2.1 the first dataset analysis

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

print("\nFirst 5 rows:")
display(df.head())

print("\nNumerical summary:")
display(df.describe())

Shape: (100775, 51)

Columns:
['id_x', 'rel_url', 'estate_rel_url_x', 'datetime_scrape_x', 'price', 'currency_x', 'location', 'attributes', 'city_when', 'city', 'day_x', 'hour_x', 'repair', 'vip', 'featured', 'products_label', 'bill_of_sale', 'mortgage', 'img_url', 'id_y', 'estate_id', 'estate_rel_url_y', 'datetime_scrape_y', 'description', 'unit_price', 'total_price', 'currency_y', 'owner_name', 'owner_title', 'shop_name', 'shop_title', 'address', 'lat', 'lng', 'updated', 'views', 'day_y', 'hour_y', 'estate_details_id_x', 'Binanın növü', 'Kateqoriya', 'Mərtəbə', 'Otaq sayı', 'Sahə', 'Torpaq sahəsi', 'Təmir', 'Çıxarış', 'İpoteka', 'estate_details_id_y', 'estate_rel_url', 'extra_info']

Data types:
id_x                    object
rel_url                 object
estate_rel_url_x        object
datetime_scrape_x       object
price                  float64
currency_x              object
location                object
attributes              object
city_when               object
city          

,id_x,rel_url,estate_rel_url_x,datetime_scrape_x,price,currency_x,location,attributes,city_when,city,day_x,hour_x,repair,vip,featured,products_label,bill_of_sale,mortgage,img_url,id_y,estate_id,estate_rel_url_y,datetime_scrape_y,description,unit_price,total_price,currency_y,owner_name,owner_title,shop_name,shop_title,address,lat,lng,updated,views,day_y,hour_y,estate_details_id_x,Binanın növü,Kateqoriya,Mərtəbə,Otaq sayı,Sahə,Torpaq sahəsi,Təmir,Çıxarış,İpoteka,estate_details_id_y,estate_rel_url,extra_info
0,5df36281-6dc6-4d5d-89a7-5fcfa86f7608,/alqi-satqi?page=174,/items/4521724,2024-10-05 22:07:37.60613+00,499999.0,AZN,Səbail r.,"4 otaqlı, 145 m², 7/9 mərtəbə","Bakı, dünən 23:52",bakı,05.10.2024,23:52,Təmirli,vipped,featured,NaN,Çıxarış var,NaN,https://bina.azstatic.com/uploads/f460x345/202...,92e82ea2-e1f3-4c2d-a284-151efd99281e,5df36281-6dc6-4d5d-89a7-5fcfa86f7608,/items/4521724,2024-10-05 22:14:10.089116+00,"Səbail Rayonu, İzzət Nəbiyev küçəsi, Fəxri Xiy...",3 450 AZN/m²,499999.0,AZN,Kamran,mülkiyyətçi,NaN,NaN,İzzət Nəbiyev küç.,40.358817,49.824092,yeniləndi: dünən 23:52,1155,05.10.2024,23:52,92e82ea2-e1f3-4c2d-a284-151efd99281e,NaN,Köhnə tikili,7 / 9,4.0,145 m²,NaN,var,var,NaN,92e82ea2-e1f3-4c2d-a284-151efd99281e,/items/4521724,Şəhidlər xiyabanı * Dağüstü parkı * Səbail r.
1,883e20f0-8872-49a5-8b4b-63b8301b5f8f,/alqi-satqi?page=250,/items/4669294,2024-10-05 22:07:37.60613+00,77000.0,AZN,Biləcəri q.,"4 otaqlı, 90 m²","Bakı, dünən 23:56",bakı,05.10.2024,23:56,Təmirli,NaN,NaN,NaN,NaN,NaN,https://bina.azstatic.com/uploads/f460x345/202...,505eaf81-6bc8-4094-9b00-aa82066548ee,883e20f0-8872-49a5-8b4b-63b8301b5f8f,/items/4669294,2024-10-05 22:14:10.089116+00,"Biləcəridə Abidəyə yaxin 91,92,202 saylı marşr...",NaN,77000.0,AZN,Dasinmaz Emlak,vasitəçi (agent),NaN,NaN,Biləcəri qəs.,40.420897,49.807035,yeniləndi: 04 oktyabr 2024,218,04.10.2024,NaN,505eaf81-6bc8-4094-9b00-aa82066548ee,NaN,Həyət evi/Bağ evi,NaN,4.0,90 m²,1.3 sot,var,yoxdur,NaN,505eaf81-6bc8-4094-9b00-aa82066548ee,/items/4669294,Binəqədi r.* Biləcəri q.
2,55c36fb1-a3af-476e-ba17-a81f6795be8d,/alqi-satqi?page=250,/items/4669293,2024-10-05 22:07:37.60613+00,92000.0,AZN,İnşaatçılar m.,"3 otaqlı, 60 m²","Bakı, dünən 23:55",bakı,05.10.2024,23:55,Təmirli,NaN,NaN,NaN,Çıxarış var,NaN,https://bina.azstatic.com/uploads/f460x345/202...,fa63b201-999d-43b5-a61a-778d9d79a6c6,55c36fb1-a3af-476e-ba17-a81f6795be8d,/items/4669293,2024-10-05 22:14:10.089116+00,Salam əleykum. \nİnşaatçılar metrosuna yaxın m...,NaN,92000.0,AZN,Məhəmməd,vasitəçi (agent),NaN,NaN,Mirzə Cabbar Məmmədzadə küç.,40.390293,49.802656,yeniləndi: 04 oktyabr 2024,190,04.10.2024,NaN,fa63b201-999d-43b5-a61a-778d9d79a6c6,NaN,Həyət evi/Bağ evi,NaN,3.0,60 m²,0.1 sot,var,var,NaN,fa63b201-999d-43b5-a61a-778d9d79a6c6,/items/4669293,İnşaatçılar m.* Yasamal r.
3,acf1aa8d-a46a-40f5-b6f2-f7a449569337,/alqi-satqi?page=250,/items/4647811,2024-10-05 22:07:37.60613+00,95000.0,AZN,Qaraçuxur q.,130 m²,"Bakı, dünən 23:55",bakı,05.10.2024,23:55,Təmirli,vipped,featured,NaN,Çıxarış var,İpoteka var,https://bina.azstatic.com/uploads/f460x345/202...,5db56980-05cc-4925-b55b-f58fb3f4d2b6,acf1aa8d-a46a-40f5-b6f2-f7a449569337,/items/4647811,2024-10-05 22:14:10.089116+00,Barter maraqlidir üstünlük maşina verilir\nHər...,NaN,95000.0,AZN,Elçin,mülkiyyətçi,NaN,NaN,Qaraçuxur qəs.,40.393614,49.981553,yeniləndi: 04 oktyabr 2024,1314,04.10.2024,NaN,5db56980-05cc-4925-b55b-f58fb3f4d2b6,NaN,Obyekt,NaN,NaN,130 m²,NaN,var,var,var,5db56980-05cc-4925-b55b-f58fb3f4d2b6,/items/4647811,Suraxanı r.* Qaraçuxur q.
4,22d840df-9283-4112-bc71-7432511fc776,/alqi-satqi?page=250,/items/4638863,2024-10-05 22:07:37.60613+00,220000.0,AZN,Əhmədli m.,"3 otaqlı, 100 m², 15/16 mərtəbə","Bakı, dünən 23:52",bakı,05.10.2024,23:52,Təmirli,NaN,NaN,Agentlik,Çıxarış var,NaN,https://bina.azstatic.com/uploads/f460x345/202...,d71cf9a9-86dd-4162-b540-6252a8659a09,22d840df-9283-4112-bc71-7432511fc776,/items/4638863,2024-10-05 22:14:10.089116+00,Əhmədli qəs. Qaçaq Nəbi küçəsi 3 otaga duzelm...


Numerical summary:


,price,total_price,lat,lng,views,Otaq sayı
count,1.007750e+05,1.007750e+05,100775.000000,100775.000000,100775.000000,91363.000000
mean,3.423557e+05,3.423553e+05,40.410846,49.886371,700.065185,3.133468
std,2.042627e+06,2.042627e+06,0.083193,0.265008,1680.672574,1.372981
min,1.100000e+01,1.100000e+01,32.689217,12.591688,23.000000,1.000000
25%,1.450000e+05,1.450000e+05,40.381900,49.816470,99.000000,2.000000
50%,2.180000e+05,2.180000e+05,40.397219,49.852867,254.000000,3.000000
75%,3.380000e+05,3.380000e+05,40.420578,49.948995,680.000000,4.000000
max,6.000000e+08,6.000000e+08,41.774687,50.341059,113458.000000,20.000000


In [125]:
# here data has many listings and missing also repeated information, the price values also vary a lot

In [126]:

# 2.2 Check price and total_price

#Because i wanted to see if they contain the same information?

# check if price and total_price are the same? 
price_equal_rate = (df["price"] == df["total_price"]).mean()

print("Percentage where price = total_price:", price_equal_rate)

# is there difference between the two columns?
print("\nDifference between price and total_price:")
print((df["price"] - df["total_price"]).describe())

Percentage where price = total_price: 0.9998709997519226

Difference between price and total_price:
count    100775.000000
mean          0.470355
std          71.974174
min       -1600.000000
25%           0.000000
50%           0.000000
75%           0.000000
max       20000.000000
dtype: float64


In [127]:
# so from output we see thgat, they are almost identical, so total_price can be removed

In [128]:
# 2.3 Checking _x and _y columns
# for same reason, to see if they contain same info

pairs = [
    ("id_x", "id_y"),
    ("estate_rel_url_x", "estate_rel_url_y"),
    ("datetime_scrape_x", "datetime_scrape_y"),
    ("currency_x", "currency_y"),
    ("estate_details_id_x", "estate_details_id_y"),
    ("estate_rel_url_x", "estate_rel_url")
]

for col1, col2 in pairs:
    same = (df[col1].fillna("") == df[col2].fillna("")).mean()
    print(col1, "vs", col2, ":", round(same * 100, 2), "% identical")

id_x vs id_y : 0.0 % identical
estate_rel_url_x vs estate_rel_url_y : 100.0 % identical
datetime_scrape_x vs datetime_scrape_y : 0.0 % identical
currency_x vs currency_y : 100.0 % identical
estate_details_id_x vs estate_details_id_y : 100.0 % identical
estate_rel_url_x vs estate_rel_url : 100.0 % identical


In [129]:
# I looked at the _x and _y columns to check for duplicated information
# 4 of them are 100% identical, but id and scrape time are different

In [130]:
# 2.4 analysing extreme prices

# there are some extreme prices, so if they are errors or real properties:

print("Prices below 1,000 AZN:")
display(
    df[df["price"] < 1000][
        ["price", "location", "attributes", "Sahə", "Otaq sayı"]
    ].head(20)
)

print("\nPrices above 10 million AZN:")
display(
    df[df["price"] > 10_000_000][
        ["price", "location", "attributes", "Sahə", "Otaq sayı"]
    ].head(20)
)

print("\nBelow 1,000 AZN:", (df["price"] < 1000).sum())
print("Above 10 million AZN:", (df["price"] > 10_000_000).sum())

Prices below 1,000 AZN:


,price,location,attributes,Sahə,Otaq sayı
8321,800.0,Zirə q.,2500 sot,2500 sot,NaN
11322,400.0,Saray q.,"2 otaqlı, 86 m², 7/14 mərtəbə",86 m²,2.0
12256,999.0,Memar Əcəmi m.,"3 otaqlı, 133 m², 9/16 mərtəbə",133 m²,3.0
15608,400.0,Saray q.,"2 otaqlı, 86 m², 7/14 mərtəbə",86 m²,2.0
16148,999.0,Memar Əcəmi m.,"3 otaqlı, 133 m², 9/16 mərtəbə",133 m²,3.0
21745,250.0,Əhmədli m.,"2 otaqlı, 30 m²",30 m²,2.0
22304,73.0,Məmmədli q.,"3 otaqlı, 90 m², 1/1 mərtəbə",90 m²,3.0
38666,11.0,Ağ şəhər q.,89 m²,89 m²,NaN
39396,700.0,Nərimanov r.,"2 otaqlı, 65 m², 2/5 mərtəbə",65 m²,2.0
39775,127.0,Sahil q.,"5 otaqlı, 150 m²",150 m²,5.0



Prices above 10 million AZN:


,price,location,attributes,Sahə,Otaq sayı
2704,20000000.0,Həzi Aslanov m.,4300 m²,4300 m²,NaN
2855,13500000.0,Nəriman Nərimanov m.,600 sot,600 sot,NaN
2922,15000000.0,Ulduz m.,160 sot,160 sot,NaN
2942,15000000.0,Nəriman Nərimanov m.,450 sot,450 sot,NaN
6151,25000000.0,Ağ şəhər q.,200 sot,200 sot,NaN
6605,600000000.0,Nəriman Nərimanov m.,"6 otaqlı, 200 m², 7/9 mərtəbə",200 m²,6.0
6942,60000000.0,5-ci mikrorayon q.,"7 otaqlı, 50 m², 6/9 mərtəbə",50 m²,7.0
7639,11000000.0,Gənclik m.,3800 m²,3800 m²,NaN
8065,36000000.0,Ağ şəhər q.,3000 m²,3000 m²,NaN
9096,18500000.0,Sahil m.,116 m²,116 m²,NaN



Below 1,000 AZN: 10
Above 10 million AZN: 101


In [131]:
# from the oyutput , we can see some unusual values are related to large land areas
# plus there are also unusual prices

In [132]:
# 2.5 Repeated Property Listings

# We check repeated listings to see if the same property appears more than once

print("Duplicate estate IDs:", df["estate_id"].duplicated().sum())

print("Duplicate property URLs:", df["estate_rel_url"].duplicated().sum())

url = df["estate_rel_url"].value_counts().idxmax()

print("\nMost repeated property URL:")
print(url)

display(
    df[df["estate_rel_url"] == url][
        ["estate_rel_url", "estate_id", "price", "Sahə", "Otaq sayı", "location"]
    ]
)

print("\nScraping dates for repeated property:")
display(
    df[df["estate_rel_url"] == url][
        ["estate_rel_url", "estate_id", "price",
         "datetime_scrape_x", "datetime_scrape_y"]
    ]
)

url_counts = df["estate_rel_url"].value_counts()

print("\nProperties appearing more than once:", (url_counts > 1).sum())

print("Maximum appearances of one property:", url_counts.max())

price_changes = df.groupby("estate_rel_url")["price"].nunique()

print("\nProperties with different prices:", (price_changes > 1).sum())

print("Properties with the same price:", (price_changes == 1).sum())

repeated_rows = df["estate_rel_url"].duplicated(keep=False).sum()

print("\nRows belonging to repeated properties:", repeated_rows)

print("Unique properties:", df["estate_rel_url"].nunique())

print(
    "Rows if one row per property were kept:",
    df.drop_duplicates("estate_rel_url").shape[0]
)

Duplicate estate IDs: 0
Duplicate property URLs: 36321

Most repeated property URL:
/items/4582627


,estate_rel_url,estate_id,price,Sahə,Otaq sayı,location
267,/items/4582627,33bd12e6-cb08-4c6e-8542-ef95bba99f79,148000.0,70 m²,3.0,20 Yanvar m.
6852,/items/4582627,ee281c6f-9b0f-4969-85c9-2ec1d43e9e86,148000.0,70 m²,3.0,20 Yanvar m.
10284,/items/4582627,7f6a797b-8717-4deb-b05a-1e0e27de3b18,148000.0,70 m²,3.0,20 Yanvar m.
14933,/items/4582627,c3b239f8-5cb7-4a09-971d-00135a5b8cfd,148000.0,70 m²,3.0,20 Yanvar m.
21054,/items/4582627,1f798d94-8a03-42c9-a4b9-19fbc5c897d9,148000.0,70 m²,3.0,20 Yanvar m.
29189,/items/4582627,d9e3e646-88a4-4b9c-8082-618c6171988f,148000.0,70 m²,3.0,20 Yanvar m.
41446,/items/4582627,460737ba-22e1-410a-97b4-13258224e3d3,148000.0,70 m²,3.0,20 Yanvar m.
47869,/items/4582627,6feef4cd-bd86-44de-9a82-7cb48d6ca021,148000.0,70 m²,3.0,20 Yanvar m.
55741,/items/4582627,1d1bfad0-4310-4016-b713-e221015ab9df,148000.0,70 m²,3.0,20 Yanvar m.
56791,/items/4582627,6a941b52-42f5-4ad9-8d74-98018f08a81b,145000.0,70 m²,3.0,20 Yanvar m.



Scraping dates for repeated property:


,estate_rel_url,estate_id,price,datetime_scrape_x,datetime_scrape_y
267,/items/4582627,33bd12e6-cb08-4c6e-8542-ef95bba99f79,148000.0,2024-10-05 22:07:37.60613+00,2024-10-05 22:14:10.089116+00
6852,/items/4582627,ee281c6f-9b0f-4969-85c9-2ec1d43e9e86,148000.0,2024-10-08 07:32:19.961753+00,2024-10-08 07:35:53.717949+00
10284,/items/4582627,7f6a797b-8717-4deb-b05a-1e0e27de3b18,148000.0,2024-10-09 04:58:40.151251+00,2024-10-09 05:02:29.349209+00
14933,/items/4582627,c3b239f8-5cb7-4a09-971d-00135a5b8cfd,148000.0,2024-10-09 22:00:47.854254+00,2024-10-09 22:05:18.232268+00
21054,/items/4582627,1f798d94-8a03-42c9-a4b9-19fbc5c897d9,148000.0,2024-10-12 05:50:17.512957+00,2024-10-12 05:53:49.164398+00
29189,/items/4582627,d9e3e646-88a4-4b9c-8082-618c6171988f,148000.0,2024-10-13 20:58:00.666852+00,2024-10-13 21:02:02.025003+00
41446,/items/4582627,460737ba-22e1-410a-97b4-13258224e3d3,148000.0,2024-10-17 23:53:36.973072+00,2024-10-17 23:58:16.367958+00
47869,/items/4582627,6feef4cd-bd86-44de-9a82-7cb48d6ca021,148000.0,2024-10-19 22:03:55.353416+00,2024-10-19 22:07:51.907099+00
55741,/items/4582627,1d1bfad0-4310-4016-b713-e221015ab9df,148000.0,2024-10-21 20:44:09.129440,2024-10-21 20:49:37.939165
56791,/items/4582627,6a941b52-42f5-4ad9-8d74-98018f08a81b,145000.0,2024-10-29 20:44:40.173239,2024-10-29 21:02:34.033173



Properties appearing more than once: 20506
Maximum appearances of one property: 17

Properties with different prices: 3356
Properties with the same price: 61098

Rows belonging to repeated properties: 56827
Unique properties: 64454
Rows if one row per property were kept: 64454


In [133]:
#from the output we see, many properties are more than one because they were scraped on different dates. and prices can change with time, 
#I kept these listings together and will use the URL as a group during model validation

In [134]:
# 2.6 Checking unit_price

# We check unit_price values to understand how this column is recorded.

print(df["unit_price"].dropna().head(20).tolist())

print("\nRandom examples:")
print(
    df["unit_price"]
    .dropna()
    .sample(20, random_state=42)
    .tolist()
)

['3 450 AZN/m²', '2 200 AZN/m²', '5 000 AZN/m²', '3 700 AZN/m²', '1 820 AZN/m²', '2 520 AZN/m²', '1 800 AZN/m²', '2 060 AZN/m²', '3 480 AZN/m²', '2 140 AZN/m²', '1 070 AZN/m²', '2 020 AZN/m²', '3 340 AZN/m²', '1 250 AZN/m²', '1 210 AZN/m²', '1 740 AZN/m²', '2 290 AZN/m²', '2 620 AZN/m²', '1 900 AZN/m²', '2 030 AZN/m²']

Random examples:
['2 450 AZN/m²', '1 930 AZN/m²', '4 800 AZN/m²', '3 120 AZN/m²', '2 450 AZN/m²', '2 320 AZN/m²', '2 040 AZN/m²', '2 650 AZN/m²', '3 040 AZN/m²', '1 670 AZN/m²', '2 460 AZN/m²', '2 150 AZN/m²', '2 460 AZN/m²', '1 930 AZN/m²', '1 850 AZN/m²', '2 500 AZN/m²', '3 000 AZN/m²', '2 420 AZN/m²', '3 200 AZN/m²', '3 520 AZN/m²']


In [135]:
# here unit_price is as text format, so its not suitable for modelling

In [136]:
# 2.7 Prepare data for modeling

# We remove columns that are not useful for the model or may cause leakage

drop_cols = [
    "total_price",
    "unit_price",
    "estate_rel_url_y",
    "currency_y",
    "estate_details_id_y",
    "datetime_scrape_y",
    "id_x",
    "id_y",
    "estate_id",
    "estate_details_id_x",
    "rel_url",
    "img_url"
]

df_model = df.drop(
    columns=drop_cols,
    errors="ignore"
).copy()

print("Shape after removing unnecessary columns:", df_model.shape)

print("\nRemaining columns:")
print(df_model.columns.tolist())

Shape after removing unnecessary columns: (100775, 39)

Remaining columns:
['estate_rel_url_x', 'datetime_scrape_x', 'price', 'currency_x', 'location', 'attributes', 'city_when', 'city', 'day_x', 'hour_x', 'repair', 'vip', 'featured', 'products_label', 'bill_of_sale', 'mortgage', 'description', 'owner_name', 'owner_title', 'shop_name', 'shop_title', 'address', 'lat', 'lng', 'updated', 'views', 'day_y', 'hour_y', 'Binanın növü', 'Kateqoriya', 'Mərtəbə', 'Otaq sayı', 'Sahə', 'Torpaq sahəsi', 'Təmir', 'Çıxarış', 'İpoteka', 'estate_rel_url', 'extra_info']


In [137]:
#so , I removed columns that were duplicated, identifiers, or could cause data leakage (kept only sueful ones)

In [138]:
# 2.8 Checking missing values

# We check missing values to decide how they should be handled before modeling.

missing_percent = (
    df_model.isnull().mean() * 100
).sort_values(ascending=False)

print("Missing values (%):")
display(missing_percent)

Missing values (%):


Binanın növü         99.847184
featured             96.875217
vip                  91.470107
Torpaq sahəsi        84.548747
hour_y               84.489209
İpoteka              67.314314
mortgage             67.314314
shop_title           28.535847
shop_name            28.535847
products_label       27.928554
Mərtəbə              24.588440
bill_of_sale         21.046887
repair               19.015629
Otaq sayı             9.339618
Təmir                 5.954850
owner_name            0.651947
owner_title           0.651947
description           0.262962
location              0.000000
price                 0.000000
datetime_scrape_x     0.000000
estate_rel_url_x      0.000000
currency_x            0.000000
city_when             0.000000
attributes            0.000000
lat                   0.000000
day_x                 0.000000
hour_x                0.000000
address               0.000000
city                  0.000000
lng                   0.000000
Kateqoriya            0.000000
updated 

In [139]:
# here we see some columns have a very high amount of missing data

In [140]:
# 2.9 Remove columns with too many missing values

# We remove columns with more than 80% missing values ( threshold)
high_missing_cols = (
    missing_percent[
        missing_percent > 80
    ]
    .index
    .tolist()
)

print("Features with more than 80% missing values:")
print(high_missing_cols)

df_model = df_model.drop(
    columns=high_missing_cols
)

print(
    "\nShape after removing columns with too many missing values:",
    df_model.shape
)

Features with more than 80% missing values:
['Binanın növü', 'featured', 'vip', 'Torpaq sahəsi', 'hour_y']

Shape after removing columns with too many missing values: (100775, 34)


In [141]:
# I removed features with more than 80% missing values

In [142]:
# 2.10 Check duplicate categorical features

# We check these columns to see if the English and Azerbaijani versions contain the same information

print("\nrepair vs Təmir:")
display(
    pd.crosstab(
        df_model["repair"].fillna("__MISSING__"),
        df_model["Təmir"].fillna("__MISSING__"),
        dropna=False
    )
)

print("\nmortgage vs İpoteka:")
display(
    pd.crosstab(
        df_model["mortgage"].fillna("__MISSING__"),
        df_model["İpoteka"].fillna("__MISSING__"),
        dropna=False
    )
)

print("\nbill_of_sale vs Çıxarış:")
display(
    pd.crosstab(
        df_model["bill_of_sale"].fillna("__MISSING__"),
        df_model["Çıxarış"].fillna("__MISSING__"),
        dropna=False
    )
)

# Remove the duplicate columns
duplicate_categorical_cols = [
    "repair",
    "mortgage",
    "bill_of_sale"
]

df_model = df_model.drop(
    columns=duplicate_categorical_cols
)

print(
    "\nShape after removing duplicate categorical columns:",
    df_model.shape
)


repair vs Təmir:


Təmir,__MISSING__,var,yoxdur
repair,,,
Təmirli,0,81612,0
__MISSING__,6001,0,13162



mortgage vs İpoteka:


İpoteka,__MISSING__,var
mortgage,,
__MISSING__,67836,0
İpoteka var,0,32939



bill_of_sale vs Çıxarış:


Çıxarış,var,yoxdur
bill_of_sale,,
__MISSING__,0,21210
Çıxarış var,79565,0



Shape after removing duplicate categorical columns: (100775, 31)


In [143]:
# I kept Azerbaijani columns and removed the duplicates

In [144]:
# 2.11 Check missing values again

# We check again to see what missing values are still left after cleaning

missing_pct = (
    df_model.isnull().mean() * 100
).sort_values(ascending=False)

display(
    missing_pct[
        missing_pct > 0
    ]
)

İpoteka           67.314314
shop_title        28.535847
shop_name         28.535847
products_label    27.928554
Mərtəbə           24.588440
Otaq sayı          9.339618
Təmir              5.954850
owner_title        0.651947
owner_name         0.651947
description        0.262962
dtype: float64

In [145]:
# in this output we see, some missing values still remain

In [146]:
# 2.12 Inspect remaining missing values

# We look at the remaining missing values to decide how to handle each column

missing_cols = df_model.columns[
    df_model.isnull().any()
]

for col in missing_cols:
    print(f"\n===== {col} =====")
    display(
        df_model[col]
        .value_counts(
            dropna=False
        )
        .head(10)
    )


===== products_label =====


products_label
Agentlik    71981
NaN         28145
Kompleks      649
Name: count, dtype: int64


===== description =====


description
NaN                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         


===== owner_name =====


...

Otaq sayı
3.0     36191
2.0     27435
4.0     15673
NaN      9412
5.0      4850
1.0      2618
6.0      2306
7.0       978
8.0       582
10.0      247
Name: count, dtype: int64


===== Təmir =====


Təmir
var       81612
yoxdur    13162
NaN        6001
Name: count, dtype: int64


===== İpoteka =====


İpoteka
NaN    67836
var    32939
Name: count, dtype: int64

In [147]:
# 2.13 Check area units

# We check the area units to make sure the values are comparable

df_model["area_unit"] = (
    df_model["Sahə"]
    .str.extract(
        r"(m²|m2|sot)",
        expand=False
    )
)

print(df_model["area_unit"].value_counts(dropna=False))

area_unit
m²     95856
sot     4919
Name: count, dtype: int64


In [148]:
# from here we see that the data contains both m² and sot

In [149]:
# 2.14 Check missing room counts

# We check missing room counts to understand why they are missing and whether they can be filled

print("Room counts:")

display(
    df_model["Otaq sayı"]
    .value_counts(dropna=False)
    .sort_index()
)

missing_rooms_by_unit = (
    df_model
    .groupby("area_unit")["Otaq sayı"]
    .apply(lambda x: x.isna().sum())
)

print("\nMissing room counts by area unit:")

display(missing_rooms_by_unit)

print("\nMissing room count for m² listings:")

display(
    df_model[
        df_model["Otaq sayı"].isna()
        &
        (df_model["area_unit"] == "m²")
    ][
        [
            "price",
            "Sahə",
            "location",
            "Təmir",
            "İpoteka"
        ]
    ].head(30)
)

Room counts:


Otaq sayı
1.0      2618
2.0     27435
3.0     36191
4.0     15673
5.0      4850
6.0      2306
7.0       978
8.0       582
9.0       241
10.0      247
11.0       43
12.0       87
13.0       10
14.0       20
15.0       35
16.0       19
18.0        4
19.0        5
20.0       19
NaN      9412
Name: count, dtype: int64


Missing room counts by area unit:


area_unit
m²     4493
sot    4919
Name: Otaq sayı, dtype: int64


Missing room count for m² listings:


,price,Sahə,location,Təmir,İpoteka
3,95000.0,130 m²,Qaraçuxur q.,var,var
8,3000000.0,485 m²,Nizami m.,var,NaN
10,1050000.0,360 m²,Masazır q.,NaN,NaN
11,275000.0,45 m²,Qara Qarayev m.,var,var
18,260000.0,70 m²,28 May m.,yoxdur,NaN
28,170000.0,45 m²,Köhnə Günəşli q.,var,NaN
34,23000.0,380 m²,Nərimanov r.,yoxdur,NaN
37,580000.0,100 m²,Əhmədli m.,var,NaN
39,600000.0,250 m²,Köhnə Günəşli q.,var,NaN
120,850000.0,150 m²,Şah İsmayıl Xətai m.,var,NaN


In [150]:
# checked the missing room counts and found that they occur in both m² and sot listings, they shouldtn be filled automatically

In [151]:
# 2.15 Convert area to numerical features

# We convert the area values to numbers so they can be used by the model

df_model["area_value"] = (
    df_model["Sahə"]
    .str.extract(
        r"([\d.]+)",
        expand=False
    )
    .astype(float)
)

df_model["area_m2"] = df_model["area_value"]

df_model.loc[
    df_model["area_unit"] == "sot",
    "area_m2"
] = (
    df_model.loc[
        df_model["area_unit"] == "sot",
        "area_value"
    ] * 100
)

display(
    df_model[
        [
            "Sahə",
            "area_unit",
            "area_value",
            "area_m2"
        ]
    ].head(20)
)

,Sahə,area_unit,area_value,area_m2
0,145 m²,m²,145.0,145.0
1,90 m²,m²,90.0,90.0
2,60 m²,m²,60.0,60.0
3,130 m²,m²,130.0,130.0
4,100 m²,m²,100.0,100.0
5,130 m²,m²,130.0,130.0
6,70 m²,m²,70.0,70.0
7,153 m²,m²,153.0,153.0
8,485 m²,m²,485.0,485.0
9,104 m²,m²,104.0,104.0


In [152]:
# I converted the area values to numerical features also changed sot values into m² 

In [153]:
# 2.16 Check missing categorical values

# We check these columns to see how many values are missing before filling them

cols_to_check = [
    "İpoteka",
    "Təmir",
    "Otaq sayı",
    "products_label"
]

for col in cols_to_check:
    print(f"\n===== {col} =====")

    print("Missing:", df_model[col].isna().sum())

    print(
        "Missing %:",
        round(df_model[col].isna().mean() * 100, 2)
    )


===== İpoteka =====
Missing: 67836
Missing %: 67.31

===== Təmir =====
Missing: 6001
Missing %: 5.95

===== Otaq sayı =====
Missing: 9412
Missing %: 9.34

===== products_label =====
Missing: 28145
Missing %: 27.93


In [154]:
# cheeking remaining missing categorical values and their percentages before filling them

In [155]:
# 2.17 Check price when values are missing

# We compare prices to see if missing values are related to the target

for col in cols_to_check:

    print(f"\n===== {col} =====")

    print("Price when value is missing:")
    display(
        df_model.loc[
            df_model[col].isna(),
            "price"
        ].describe()
    )

    print("Price when value is not missing:")
    display(
        df_model.loc[
            df_model[col].notna(),
            "price"
        ].describe()
    )


===== İpoteka =====
Price when value is missing:


count    6.783600e+04
mean     3.504351e+05
std      2.436309e+06
min      1.100000e+01
25%      1.370000e+05
50%      2.150000e+05
75%      3.400000e+05
max      6.000000e+08
Name: price, dtype: float64

Price when value is not missing:


count    3.293900e+04
mean     3.257168e+05
std      7.352607e+05
min      1.700000e+03
25%      1.550000e+05
50%      2.230000e+05
75%      3.350000e+05
max      3.600000e+07
Name: price, dtype: float64


===== Təmir =====
Price when value is missing:


...

count    9.136300e+04
mean     2.801083e+05
std      2.011273e+06
min      7.300000e+01
25%      1.450000e+05
50%      2.150000e+05
75%      3.220250e+05
max      6.000000e+08
Name: price, dtype: float64


===== products_label =====
Price when value is missing:


count    2.814500e+04
mean     3.136199e+05
std      3.662944e+06
min      1.100000e+01
25%      1.050000e+05
50%      1.670000e+05
75%      2.800000e+05
max      6.000000e+08
Name: price, dtype: float64

Price when value is not missing:


count    7.263000e+04
mean     3.534912e+05
std      7.678085e+05
min      4.000000e+02
25%      1.600000e+05
50%      2.350000e+05
75%      3.500000e+05
max      3.900000e+07
Name: price, dtype: float64

In [156]:
# there are  noticeable differences, especially for room count and repair status

In [157]:
# 2.18 Check mortgage values

# We check mortgage values to understand what the missing values may mean

print(df_model["İpoteka"].value_counts(dropna=False))

print("\nMortgage = var:")
display(
    df_model[
        df_model["İpoteka"] == "var"
    ][
        ["price", "Sahə", "location", "Otaq sayı", "Təmir"]
    ].head(20)
)

print("\nMortgage = missing:")
display(
    df_model[
        df_model["İpoteka"].isna()
    ][
        ["price", "Sahə", "location", "Otaq sayı", "Təmir"]
    ].head(20)
)

İpoteka
NaN    67836
var    32939
Name: count, dtype: int64

Mortgage = var:


,price,Sahə,location,Otaq sayı,Təmir
3,95000.0,130 m²,Qaraçuxur q.,NaN,var
11,275000.0,45 m²,Qara Qarayev m.,NaN,var
14,330000.0,160 m²,Nəriman Nərimanov m.,3.0,var
15,195000.0,56 m²,8 Noyabr m.,2.0,var
19,455000.0,225 m²,Əhmədli m.,5.0,var
27,680000.0,260 m²,Sahil m.,4.0,var
32,135000.0,100 m²,Görədil q.,4.0,var
36,143000.0,65 m²,Memar Əcəmi m.,2.0,var
38,133000.0,52 m²,Nəsimi m.,2.0,var
40,282000.0,140 m²,Nəsimi m.,3.0,var



Mortgage = missing:


,price,Sahə,location,Otaq sayı,Təmir
0,499999.0,145 m²,Səbail r.,4.0,var
1,77000.0,90 m²,Biləcəri q.,4.0,var
2,92000.0,60 m²,İnşaatçılar m.,3.0,var
4,220000.0,100 m²,Əhmədli m.,3.0,var
5,650000.0,130 m²,Sahil m.,4.0,var
6,259000.0,70 m²,Sahil m.,3.0,var
7,279000.0,153 m²,Bayıl q.,3.0,yoxdur
8,3000000.0,485 m²,Nizami m.,NaN,var
9,262000.0,104 m²,Memar Əcəmi m.,3.0,var
10,1050000.0,360 m²,Masazır q.,NaN,NaN


In [158]:
# here we see that missing mortgage values do not really mean “no mortgage,” so I kept them as missing

In [159]:
# 2.19 Check products_label

# We check products_label to understand its values and whether missing values are useful or should be handled

print(df_model["products_label"].value_counts(dropna=False))

print("\nProducts_label = missing:")
display(
    df_model[
        df_model["products_label"].isna()
    ][
        ["price", "Sahə", "location", "Otaq sayı", "Təmir", "İpoteka"]
    ].head(20)
)

print("\nProducts_label = available:")
display(
    df_model[
        df_model["products_label"].notna()
    ][
        ["price", "Sahə", "location", "Otaq sayı", "Təmir", "İpoteka"]
    ].head(20)
)

print("\nPrice when products_label is missing:")
display(
    df_model.loc[
        df_model["products_label"].isna(),
        "price"
    ].describe()
)

print("\nPrice when products_label is not missing:")
display(
    df_model.loc[
        df_model["products_label"].notna(),
        "price"
    ].describe()
)

print("\nPrice distribution by products_label:")
display(
    df_model.groupby(
        df_model["products_label"].fillna("Missing")
    )["price"].describe()
)

for col in ["location", "Təmir", "İpoteka"]:
    print(f"\n===== {col} vs products_label =====")

    display(
        pd.crosstab(
            df_model[col],
            df_model["products_label"],
            normalize="index"
        ).round(3)
    )

products_label
Agentlik    71981
NaN         28145
Kompleks      649
Name: count, dtype: int64

Products_label = missing:


,price,Sahə,location,Otaq sayı,Təmir,İpoteka
0,499999.0,145 m²,Səbail r.,4.0,var,NaN
1,77000.0,90 m²,Biləcəri q.,4.0,var,NaN
2,92000.0,60 m²,İnşaatçılar m.,3.0,var,NaN
3,95000.0,130 m²,Qaraçuxur q.,NaN,var,var
5,650000.0,130 m²,Sahil m.,4.0,var,NaN
6,259000.0,70 m²,Sahil m.,3.0,var,NaN
7,279000.0,153 m²,Bayıl q.,3.0,yoxdur,NaN
8,3000000.0,485 m²,Nizami m.,NaN,var,NaN
10,1050000.0,360 m²,Masazır q.,NaN,NaN,NaN
11,275000.0,45 m²,Qara Qarayev m.,NaN,var,var



Products_label = available:


,price,Sahə,location,Otaq sayı,Təmir,İpoteka
4,220000.0,100 m²,Əhmədli m.,3.0,var,NaN
9,262000.0,104 m²,Memar Əcəmi m.,3.0,var,NaN
12,153000.0,85 m²,İnşaatçılar m.,4.0,var,NaN
14,330000.0,160 m²,Nəriman Nərimanov m.,3.0,var,var
15,195000.0,56 m²,8 Noyabr m.,2.0,var,var
27,680000.0,260 m²,Sahil m.,4.0,var,var
28,170000.0,45 m²,Köhnə Günəşli q.,NaN,var,NaN
29,95000.0,50 m²,Neftçilər m.,2.0,var,NaN
34,23000.0,380 m²,Nərimanov r.,NaN,yoxdur,NaN
36,143000.0,65 m²,Memar Əcəmi m.,2.0,var,var



Price when products_label is missing:


...

products_label,Agentlik,Kompleks
location,,
2-ci Alatava q.,1.000,0.000
2-ci mikrorayon q.,1.000,0.000
20 Yanvar m.,0.977,0.023
20-ci sahə q.,1.000,0.000
28 May m.,0.992,0.008
...,...,...
Şərq q.,1.000,0.000
Əhmədli m.,1.000,0.000
Əhmədli q.,1.000,0.000



===== Təmir vs products_label =====


products_label,Agentlik,Kompleks
Təmir,,
var,0.996,0.004
yoxdur,0.958,0.042



===== İpoteka vs products_label =====


products_label,Agentlik,Kompleks
İpoteka,,
var,0.986,0.014


In [160]:
# we see that most listings are `Agentlik`, while missing values are common, so I kept the missing category

In [161]:
# 2.20 Fill categorical missing values

# We fill missing categorical values with "Missing" so we do not lose these rows

for col in [
    "İpoteka",
    "Təmir",
    "Çıxarış",
    "products_label"
]:

    df_model[col] = (
        df_model[col]
        .fillna("Missing")
    )

print("\nİpoteka:")
print(df_model["İpoteka"].value_counts())

print("\nTəmir:")
print(df_model["Təmir"].value_counts())

print("\nÇıxarış:")
print(df_model["Çıxarış"].value_counts())

print("\nproducts_label:")
print(df_model["products_label"].value_counts())


İpoteka:
İpoteka
Missing    67836
var        32939
Name: count, dtype: int64

Təmir:
Təmir
var        81612
yoxdur     13162
Missing     6001
Name: count, dtype: int64

Çıxarış:
Çıxarış
var       79565
yoxdur    21210
Name: count, dtype: int64

products_label:
products_label
Agentlik    71981
Missing     28145
Kompleks      649
Name: count, dtype: int64


In [162]:
# Missing values replaced with "Missing" to keep all rows

In [163]:
# 2.21 Seller / Agency Features

# We remove seller and agency columns because they are mostly identifiers and are not useful for predicting price

print(
    df_model[
        [
            "owner_name",
            "owner_title",
            "shop_name",
            "shop_title"
        ]
    ].isnull().sum()
)

seller_cols = [
    "owner_name",
    "shop_name",
    "shop_title"
]

df_model = df_model.drop(
    columns=seller_cols
)

print(
    "Shape after removing seller/agency columns:",
    df_model.shape
)

owner_name       657
owner_title      657
shop_name      28757
shop_title     28757
dtype: int64
Shape after removing seller/agency columns: (100775, 31)


In [164]:
# Seller and agency columns were removed because they dont provide useful information about the property price

In [165]:
# 2.22 Floor Features

# We split the floor information into separate numbers so the model can use them

print("Unique Mərtəbə values:", df_model["Mərtəbə"].nunique())

display(
    df_model["Mərtəbə"]
    .dropna()
    .head(20)
)

floor_parts = (
    df_model["Mərtəbə"]
    .str.split("/", expand=True)
)

df_model["floor"] = pd.to_numeric(
    floor_parts[0],
    errors="coerce"
)

df_model["total_floors"] = pd.to_numeric(
    floor_parts[1],
    errors="coerce"
)

display(
    df_model[
        [
            "Mərtəbə",
            "floor",
            "total_floors"
        ]
    ].head(20)
)

Unique Mərtəbə values:

 392


0       7 / 9
4     15 / 16
5       3 / 4
6       2 / 5
7      3 / 18
9     11 / 13
12    19 / 20
14     3 / 16
15    10 / 16
16      4 / 5
17      5 / 6
19    12 / 16
20    16 / 18
21      3 / 6
23      5 / 6
24     9 / 16
25      8 / 9
27    13 / 17
29      5 / 5
31     4 / 17
Name: Mərtəbə, dtype: object

,Mərtəbə,floor,total_floors
0,7 / 9,7.0,9.0
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,15 / 16,15.0,16.0
5,3 / 4,3.0,4.0
6,2 / 5,2.0,5.0
7,3 / 18,3.0,18.0
8,NaN,NaN,NaN
9,11 / 13,11.0,13.0


In [166]:
# I split the floor information into floor and total_floors so the model can use them as numerical features

In [167]:
# 2.23 Price per m² — Exploratory Analysis Only

# We calculate price per m² to better understand the relationship between price and area

df_model["price_per_m2"] = np.where(
    df_model["area_m2"] > 0,
    df_model["price"] / df_model["area_m2"],
    np.nan
)

display(
    df_model[
        [
            "price",
            "Sahə",
            "area_m2",
            "price_per_m2"
        ]
    ].head(20)
)

display(
    df_model["price_per_m2"].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

,price,Sahə,area_m2,price_per_m2
0,499999.0,145 m²,145.0,3448.268966
1,77000.0,90 m²,90.0,855.555556
2,92000.0,60 m²,60.0,1533.333333
3,95000.0,130 m²,130.0,730.769231
4,220000.0,100 m²,100.0,2200.000000
5,650000.0,130 m²,130.0,5000.000000
6,259000.0,70 m²,70.0,3700.000000
7,279000.0,153 m²,153.0,1823.529412
8,3000000.0,485 m²,485.0,6185.567010
9,262000.0,104 m²,104.0,2519.230769


count    1.007750e+05
mean     2.321759e+03
std      1.042945e+04
min      2.500000e-04
1%       4.900000e+01
5%       4.500000e+02
25%      1.675000e+03
50%      2.214286e+03
75%      2.730496e+03
95%      3.827777e+03
99%      6.666667e+03
max      3.000000e+06
Name: price_per_m2, dtype: float64

In [168]:
# I calculated price per m² to explore how property price related to area and identify unusual values

In [169]:
# 2.24 Extreme Price per m² Values

# We check extreme price per m² values to see if they look unusual or incorrect

print("Highest price/m² observations:")

display(
    df_model.nlargest(
        20,
        "price_per_m2"
    )[
        [
            "price",
            "Sahə",
            "area_unit",
            "area_m2",
            "price_per_m2",
            "location"
        ]
    ]
)

print("Lowest price/m² observations:")

display(
    df_model.nsmallest(
        20,
        "price_per_m2"
    )[
        [
            "price",
            "Sahə",
            "area_unit",
            "area_m2",
            "price_per_m2",
            "location"
        ]
    ]
)

Highest price/m² observations:


,price,Sahə,area_unit,area_m2,price_per_m2,location
6605,600000000.0,200 m²,m²,200.0,3.000000e+06,Nəriman Nərimanov m.
6942,60000000.0,50 m²,m²,50.0,1.200000e+06,5-ci mikrorayon q.
12285,380000.0,1 m²,m²,1.0,3.800000e+05,Şüvəlan q.
16168,380000.0,1 m²,m²,1.0,3.800000e+05,Şüvəlan q.
9096,18500000.0,116 m²,m²,116.0,1.594828e+05,Sahil m.
14281,18500000.0,116 m²,m²,116.0,1.594828e+05,Sahil m.
11364,650000.0,9 m²,m²,9.0,7.222222e+04,Azadlıq Prospekti m.
15638,650000.0,9 m²,m²,9.0,7.222222e+04,Azadlıq Prospekti m.
34426,5000000.0,98 m²,m²,98.0,5.102041e+04,Sahil m.
98699,5000000.0,98 m²,m²,98.0,5.102041e+04,Sahil m.


Lowest price/m² observations:


,price,Sahə,area_unit,area_m2,price_per_m2,location
4736,2500.0,100000 sot,sot,10000000.0,0.000250,Fatmayı q.
9746,2500.0,100000 sot,sot,10000000.0,0.000250,Fatmayı q.
21517,2500.0,100000 sot,sot,10000000.0,0.000250,Fatmayı q.
75857,3000.0,100000 sot,sot,10000000.0,0.000300,Fatmayı q.
8321,800.0,2500 sot,sot,250000.0,0.003200,Zirə q.
3348,3500.0,4000 sot,sot,400000.0,0.008750,Buzovna q.
89280,1000.0,200 sot,sot,20000.0,0.050000,Şıxov q.
75936,30000.0,3500 sot,sot,350000.0,0.085714,Xaçmaz
172,80000.0,7000 sot,sot,700000.0,0.114286,Maştağa q.
38666,11.0,89 m²,m²,89.0,0.123596,Ağ şəhər q.


In [170]:
# some price/m² values show some listings have incorrect prices or areas

In [171]:
# 2.25 Flag unusual price per m²

# We flag very low or high price per m² values so we can review them later

df_model["price_per_m2_flag"] = (
    (df_model["price_per_m2"] < 100)
    |
    (df_model["price_per_m2"] > 10000)
)

print(
    "Potentially unusual price/m² observations:",
    df_model["price_per_m2_flag"].sum()
)

display(
    df_model[
        df_model["price_per_m2_flag"]
    ][
        [
            "price",
            "Sahə",
            "area_unit",
            "area_value",
            "area_m2",
            "price_per_m2",
            "location"
        ]
    ]
    .sort_values("price_per_m2")
    .head(20)
)

Potentially unusual price/m² observations: 2338


,price,Sahə,area_unit,area_value,area_m2,price_per_m2,location
4736,2500.0,100000 sot,sot,100000.0,10000000.0,0.000250,Fatmayı q.
9746,2500.0,100000 sot,sot,100000.0,10000000.0,0.000250,Fatmayı q.
21517,2500.0,100000 sot,sot,100000.0,10000000.0,0.000250,Fatmayı q.
75857,3000.0,100000 sot,sot,100000.0,10000000.0,0.000300,Fatmayı q.
8321,800.0,2500 sot,sot,2500.0,250000.0,0.003200,Zirə q.
3348,3500.0,4000 sot,sot,4000.0,400000.0,0.008750,Buzovna q.
89280,1000.0,200 sot,sot,200.0,20000.0,0.050000,Şıxov q.
75936,30000.0,3500 sot,sot,3500.0,350000.0,0.085714,Xaçmaz
172,80000.0,7000 sot,sot,7000.0,700000.0,0.114286,Maştağa q.
38666,11.0,89 m²,m²,89.0,89.0,0.123596,Ağ şəhər q.


In [172]:
# I noted 2,338 listings with unusually low or high price per m² for later review

In [173]:
# 2.26 Extremely large areas

# We check very large areas to see if they are realistic or possible data errors

display(
    df_model["area_m2"].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

large_area_mask = (
    df_model["area_m2"] > 10000
)

print(
    "Properties with area > 10,000 m²:",
    large_area_mask.sum()
)

display(
    df_model[
        large_area_mask
    ][
        [
            "price",
            "Sahə",
            "area_unit",
            "area_value",
            "area_m2",
            "location"
        ]
    ]
    .sort_values(
        "area_m2",
        ascending=False
    )
    .head(30)
)

count    1.007750e+05
mean     1.128061e+03
std      9.320227e+04
min      1.000000e+00
1%       3.500000e+01
5%       4.720000e+01
25%      7.000000e+01
50%      1.050000e+02
75%      1.600000e+02
95%      6.000000e+02
99%      2.800000e+03
max      2.000000e+07
Name: area_m2, dtype: float64

Properties with area > 10,000 m²: 389


,price,Sahə,area_unit,area_value,area_m2,location
45825,40000000.0,200000 sot,sot,200000.0,20000000.0,28 May m.
75857,3000.0,100000 sot,sot,100000.0,10000000.0,Fatmayı q.
21517,2500.0,100000 sot,sot,100000.0,10000000.0,Fatmayı q.
9746,2500.0,100000 sot,sot,100000.0,10000000.0,Fatmayı q.
4736,2500.0,100000 sot,sot,100000.0,10000000.0,Fatmayı q.
69308,26000000.0,60000 sot,sot,60000.0,6000000.0,Binəqədi r.
28995,19000000.0,60000 sot,sot,60000.0,6000000.0,Binəqədi r.
172,80000.0,7000 sot,sot,7000.0,700000.0,Maştağa q.
87428,2640000.0,4400 sot,sot,4400.0,440000.0,Qobu q.
37618,16000000.0,4000 sot,sot,4000.0,400000.0,Lökbatan q.


In [174]:
# There are 389 properties larger than 10,000 m², some area values may be unrealistic or data errors

In [175]:
# 2.27 Remove temporary EDA variables

# We remove these temporary columns because they were only used for exploration

df_model = df_model.drop(
    columns=[
        "price_per_m2",
        "price_per_m2_flag"
    ],
    errors="ignore"
)

In [176]:
# 2.28 Remove original text columns

# We remove the original text columns because we already extracted the useful numerical information.

df_model = df_model.drop(
    columns=[
        "Sahə",
        "Mərtəbə"
    ],
    errors="ignore"
)

In [177]:
# 2.29 Final feature cleanup

# We remove constant or duplicate columns that do not add new information

# Remove currency if it has only one value
if (
    "currency_x" in df_model.columns
    and df_model["currency_x"].nunique(dropna=False) == 1
):
    df_model = df_model.drop(
        columns=["currency_x"]
    )

# Remove duplicate URL column if both columns are identical
if (
    "estate_rel_url_x" in df_model.columns
    and "estate_rel_url" in df_model.columns
):
    url_same = (
        df_model["estate_rel_url_x"].fillna("__MISSING__")
        ==
        df_model["estate_rel_url"].fillna("__MISSING__")
    ).mean()

    print(
        "estate_rel_url_x vs estate_rel_url:",
        f"{url_same:.2%}"
    )

    if url_same == 1.0:
        df_model = df_model.drop(
            columns=["estate_rel_url_x"]
        )

estate_rel_url_x vs estate_rel_url: 100.00%


In [178]:
# constant and duplicate columns were removed

In [179]:
# 2.30 Final dataset check

# We check the final dataset to make sure it is clean and ready for modeling

print("Final cleaned dataframe shape:", df_model.shape)

print("\nFinal columns:")
print(df_model.columns.tolist())

print("\nRemaining missing values:")
display(
    df_model.isnull()
    .sum()
    .sort_values(ascending=False)
    .head(30)
)

print("\nData types:")
print(df_model.dtypes)

Final cleaned dataframe shape: (100775, 29)

Final columns:
['datetime_scrape_x', 'price', 'location', 'attributes', 'city_when', 'city', 'day_x', 'hour_x', 'products_label', 'description', 'owner_title', 'address', 'lat', 'lng', 'updated', 'views', 'day_y', 'Kateqoriya', 'Otaq sayı', 'Təmir', 'Çıxarış', 'İpoteka', 'estate_rel_url', 'extra_info', 'area_unit', 'area_value', 'area_m2', 'floor', 'total_floors']

Remaining missing values:


total_floors         24779
floor                24779
Otaq sayı             9412
owner_title            657
description            265
city_when                0
attributes               0
city                     0
day_x                    0
products_label           0
location                 0
price                    0
datetime_scrape_x        0
lat                      0
address                  0
hour_x                   0
lng                      0
day_y                    0
Kateqoriya               0
views                    0
updated                  0
Çıxarış                  0
Təmir                    0
İpoteka                  0
estate_rel_url           0
area_unit                0
extra_info               0
area_m2                  0
area_value               0
dtype: int64


Data types:
datetime_scrape_x     object
price                float64
location              object
attributes            object
city_when             object
city                  object
day_x                 object
hour_x                object
products_label        object
description           object
owner_title           object
address               object
lat                  float64
lng                  float64
updated               object
views                  int64
day_y                 object
Kateqoriya            object
Otaq sayı            float64
Təmir                 object
Çıxarış               object
İpoteka               object
estate_rel_url        object
extra_info            object
area_unit             object
area_value           float64
area_m2              float64
floor                float64
total_floors         float64
dtype: object


In [180]:
# The final dataset contains 100,775 rows and 29 features, with only floor, room count, owner title, and description still having missing values

In [181]:
# 2.31 Final leakage / identifier check

# We check that no columns that could cause leakage or act as identifiers are left.

leakage_or_identifier_cols = [
    "total_price",
    "unit_price",
    "id_x",
    "id_y",
    "estate_id",
    "estate_rel_url_x",
    "estate_rel_url_y",
    "estate_details_id_x",
    "estate_details_id_y",
    "rel_url",
    "img_url",
    "price_per_m2",
    "price_per_m2_flag"
]

remaining_leakage_cols = [
    col
    for col in leakage_or_identifier_cols
    if col in df_model.columns
]

print(
    "\nRemaining leakage/identifier columns:",
    remaining_leakage_cols
)


Remaining leakage/identifier columns: []


In [182]:
# 2.32 Check repeated property groups

# We check property groups to understand if multiple rows belong to the same property.

if "estate_rel_url" in df_model.columns:

    groups = df_model["estate_rel_url"].copy()

    print("\nUnique property groups:", groups.nunique())
    print("Total observations:", len(groups))


Unique property groups: 64454
Total observations: 100775


In [183]:
# 2.33 Separate target and features

# We separate the target, features, and property groups before modeling

# Keep property ID for group-based splitting
groups = df_model["estate_rel_url"].copy()

# Create features and target
X = df_model.drop(
    columns=["price", "estate_rel_url"]
).copy()

y = df_model["price"].copy()

print("\nX shape:", X.shape)
print("y shape:", y.shape)
print("groups shape:", groups.shape)

print("\nTarget range:", y.min(), "-", y.max())


X shape: (100775, 27)
y shape: (100775,)
groups shape: (100775,)

Target range: 11.0 - 600000000.0


In [184]:
# 2.34 Final checks

# We check that the data is consistent and ready for the modeling stage.

assert len(X) == len(y) == len(groups)

assert "price" not in X.columns
assert "estate_rel_url" not in X.columns

assert "price_per_m2" not in X.columns
assert "price_per_m2_flag" not in X.columns
assert "total_price" not in X.columns
assert "unit_price" not in X.columns

assert y.isnull().sum() == 0
assert (y > 0).all()


# Checkpoint 3 — Model Comparison

In this step, I compared three different models to find which one performs best for house price prediction.

I used the same training data, the same 3-fold group-based cross-validation, and the same evaluation metrics for all three models. The models were **Ridge Regression, Decision Tree, and Random Forest**.

I used **MAE, RMSE, and R²** to compare their performance. MAE is the main metric because it shows the average prediction error in AZN.

I also made sure that repeated listings of the same property stayed in the same group, so the same property could not appear in both the training and validation sets.

The model with the lowest average CV MAE was selected as the best model for the next checkpoint.


In [185]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold,
    cross_validate
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [186]:
# 3.1 Features, Target, Groups

# We select the features, target, and property groups that will be used to compare the models.

groups = df_model["estate_rel_url"].copy()

model_features = [
    "location",
    "city",
    "Kateqoriya",
    "products_label",
    "Təmir",
    "Çıxarış",
    "İpoteka",
    "owner_title",
    "area_unit",
    "area_m2",
    "area_value",
    "floor",
    "total_floors",
    "Otaq sayı",
    "lat",
    "lng",
    "views"
]

X = df_model[model_features].copy()
y = df_model["price"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("groups shape:", groups.shape)

assert len(X) == len(y) == len(groups)

X shape: (100775, 17)
y shape: (100775,)
groups shape: (100775,)


In [187]:
# The final model uses 17 features, with price as the target and estate_rel_url as the property group identifier

In [188]:
# 3.2 Group-Aware Train/Test Split

# We split the data by property group so the same property cannot appear in both sets.

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("\nX_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

train_groups = set(groups_train.unique())
test_groups = set(groups_test.unique())

overlap = train_groups.intersection(test_groups)

print("Train/test property overlap:", len(overlap))

assert len(overlap) == 0


X_train: (80688, 17)
X_test: (20087, 17)
y_train: (80688,)
y_test: (20087,)
Train/test property overlap: 0


In [189]:
# 80/20 split was used, to be sure no property appears in both training and test sets

In [190]:
# 3.3 Feature types

# We separate numerical and categorical features because they need different preprocessing.

numeric_features = [
    "area_m2",
    "area_value",
    "floor",
    "total_floors",
    "Otaq sayı",
    "lat",
    "lng",
    "views"
]

categorical_features = [
    "location",
    "city",
    "Kateqoriya",
    "products_label",
    "Təmir",
    "Çıxarış",
    "İpoteka",
    "owner_title",
    "area_unit"
]

In [191]:
# 3.4 Shared preprocessing

# We prepare numerical and categorical features in the same way for all models.

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [192]:
# 3.5 Three models

# We use three different models so we can compare their performance.

ridge_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=10.0))
    ]
)

tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeRegressor(
                max_depth=12,
                min_samples_leaf=5,
                random_state=42
            )
        )
    ]
)

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=25,
                max_depth=12,
                min_samples_leaf=5,
                random_state=42,
                n_jobs=2
            )
        )
    ]
)

models = {
    "Ridge Regression": ridge_model,
    "Decision Tree": tree_model,
    "Random Forest": random_forest_model
}

print("\nModels:")
for name in models:
    print("-", name)

assert len(models) >= 3


Models:
- Ridge Regression
- Decision Tree
- Random Forest


In [193]:
# 3.6 Same group-aware CV for all models

# We use the same group-based cross-validation for every model so the results are fair.

group_kfold = GroupKFold(
    n_splits=3
)

cv_splits = list(
    group_kfold.split(
        X_train,
        y_train,
        groups=groups_train
    )
)

print(
    "\nNumber of CV folds:",
    len(cv_splits)
)

for fold, (train_fold, val_fold) in enumerate(
    cv_splits,
    start=1
):

    train_fold_groups = set(
        groups_train.iloc[
            train_fold
        ].unique()
    )

    val_fold_groups = set(
        groups_train.iloc[
            val_fold
        ].unique()
    )

    fold_overlap = (
        train_fold_groups.intersection(
            val_fold_groups
        )
    )

    print(
        f"Fold {fold}: "
        f"train={len(train_fold)}, "
        f"validation={len(val_fold)}, "
        f"group overlap={len(fold_overlap)}"
    )

    assert len(fold_overlap) == 0


Number of CV folds: 3
Fold 1: train=53792, validation=26896, group overlap=0
Fold 2: train=53792, validation=26896, group overlap=0
Fold 3: train=53792, validation=26896, group overlap=0


In [194]:
# 3.7 Same metrics

# We use the same metrics for all models so their results can be compared fairly.

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

In [195]:
# 3.8 Evaluate models

# We evaluate each model using the same cross-validation and metrics.

cv_results = {}

for name, model in models.items():

    print(f"\nEvaluating {name}...")

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv_splits,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False
    )

    cv_results[name] = scores

    print(f"{name}: done")


Evaluating Ridge Regression...
Ridge Regression: done

Evaluating Decision Tree...
Decision Tree: done

Evaluating Random Forest...
Random Forest: done


In [196]:
# 3.9 Fold results

# We look at the results from each fold to see how consistent the models are.

fold_results = []

for name, scores in cv_results.items():

    mae = -scores["test_MAE"]
    rmse = -scores["test_RMSE"]
    r2 = scores["test_R2"]

    for fold in range(3):

        fold_results.append(
            {
                "Model": name,
                "Fold": fold + 1,
                "MAE": mae[fold],
                "RMSE": rmse[fold],
                "R2": r2[fold]
            }
        )

fold_results_df = pd.DataFrame(
    fold_results
)

display(fold_results_df)

,Model,Fold,MAE,RMSE,R2
0,Ridge Regression,1,214372.945735,7.768467e+05,0.071209
1,Ridge Regression,2,219005.137215,7.613059e+05,0.143918
2,Ridge Regression,3,202175.410146,3.728587e+06,0.005426
3,Decision Tree,1,133755.809730,1.366010e+06,-1.871804
4,Decision Tree,2,140940.268734,1.570128e+06,-2.641390
5,Decision Tree,3,136116.314272,3.706595e+06,0.017124
6,Random Forest,1,114895.026783,1.170721e+06,-1.109376
7,Random Forest,2,130039.451402,1.550115e+06,-2.549155
8,Random Forest,3,122355.422072,3.691127e+06,0.025310


In [197]:
# The fold results show that Random Forest has the lowest MAE across all three folds, making it the strongest model among all

In [198]:
# 3.10 Comparison table

# We compare the average and variation of each model's results to choose the best one.

rows = []

for name, scores in cv_results.items():

    mae = -scores["test_MAE"]
    rmse = -scores["test_RMSE"]
    r2 = scores["test_R2"]

    rows.append(
        {
            "Model": name,

            "MAE Mean": mae.mean(),
            "MAE Std": mae.std(),

            "RMSE Mean": rmse.mean(),
            "RMSE Std": rmse.std(),

            "R2 Mean": r2.mean(),
            "R2 Std": r2.std(),

            "MAE":
                f"{mae.mean():,.2f} +/- {mae.std():,.2f}",

            "RMSE":
                f"{rmse.mean():,.2f} +/- {rmse.std():,.2f}",

            "R2":
                f"{r2.mean():.4f} +/- {r2.std():.4f}"
        }
    )

comparison_df = pd.DataFrame(rows)

comparison_df = (
    comparison_df
    .sort_values("MAE Mean")
    .reset_index(drop=True)
)

display(comparison_df)

,Model,MAE Mean,MAE Std,RMSE Mean,RMSE Std,R2 Mean,R2 Std,MAE,RMSE,R2
0,Random Forest,122429.966752,6182.910153,2.137321e+06,1.109570e+06,-1.211074,1.053478,"122,429.97 +/- 6,182.91","2,137,320.91 +/- 1,109,570.14",-1.2111 +/- 1.0535
1,Decision Tree,136937.464245,2989.964131,2.214244e+06,1.058536e+06,-1.498690,1.116941,"136,937.46 +/- 2,989.96","2,214,244.26 +/- 1,058,536.38",-1.4987 +/- 1.1169
2,Ridge Regression,211851.164365,7098.331459,1.755580e+06,1.395141e+06,0.073518,0.056563,"211,851.16 +/- 7,098.33","1,755,579.83 +/- 1,395,141.13",0.0735 +/- 0.0566


In [199]:
# 3.11 Best model

# We choose the model with the lowest average MAE because lower error means better predictions.

best_model_name = comparison_df.iloc[0]["Model"]

print(
    "\nBest model based on CV MAE:",
    best_model_name
)


Best model based on CV MAE: Random Forest


In [200]:
# 3.12 Final checks

# We check that all models have valid results and the comparison is complete.

for name, scores in cv_results.items():

    for metric in [
        "test_MAE",
        "test_RMSE",
        "test_R2"
    ]:

        assert len(scores[metric]) == 3

        assert not np.isnan(
            scores[metric]
        ).any()

assert len(models) >= 3
assert len(cv_splits) == 3
assert len(overlap) == 0

assert (
    comparison_df["MAE Mean"]
    .notna()
    .all()
)

assert (
    comparison_df["RMSE Mean"]
    .notna()
    .all()
)

assert (
    comparison_df["R2 Mean"]
    .notna()
    .all()
)


## 3.11 Model Comparison and Selection

All three regression models were evaluated using the same 3-fold GroupKFold cross-validation strategy and the same evaluation metrics: MAE, RMSE, and R². The property identifier was used only for grouping, ensuring that observations from the same property did not appear in both the training and validation folds.

Random Forest achieved the lowest mean CV MAE at approximately **122,430 AZN**, compared with **136,937 AZN** for Decision Tree and **211,851 AZN** for Ridge Regression.

Since **MAE was defined as the primary evaluation metric**, Random Forest was selected as the best-performing model for the next stage. Its fold-to-fold MAE variation was also relatively small compared with the size of the prediction errors.

Ridge Regression achieved a positive mean R², while the tree-based models produced negative mean R² and very large RMSE values. This indicates that the extreme values in the target variable have a strong effect on squared-error-based metrics. Because the project prioritizes MAE, model selection is based primarily on MAE rather than RMSE or R².

The Random Forest model will therefore be carried forward to **Checkpoint 4 for hyperparameter tuning**. The test set has not been used for model selection or evaluation at this stage.

# Checkpoint 4 — Hyperparameter Tuning

In this step, I tuned the Random Forest model to try to improve its performance.

I used `RandomizedSearchCV` with the same 3-fold group-based cross-validation from Checkpoint 3. The tuning was done using **training data only**, so the test set was kept completely untouched.

I tested different values for the number of trees, tree depth, minimum samples, and number of features used by the model.

The best tuned Random Forest achieved a CV MAE of **110,468.21 AZN**, compared with **122,429.97 AZN** for the baseline Random Forest. This improved the MAE by **9.77%**.

The tuned model was selected as the final model for the next step, where it will be evaluated once on the separate test set.


In [201]:
import numpy as np
import pandas as pd

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

In [202]:
# 4.1 Recreate Random Forest Pipeline

# We recreate the Random Forest pipeline so we can tune its parameters.

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                random_state=42,
                n_jobs=2
            )
        )
    ]
)

In [203]:
# 4.2 Hyperparameter search

# We define a small range of parameters to find a better Random Forest model.

param_distributions = {
    "model__n_estimators": [
        20,
        30
    ],

    "model__max_depth": [
        8,
        12,
        16
    ],

    "model__min_samples_split": [
        2,
        5
    ],

    "model__min_samples_leaf": [
        2,
        5
    ],

    "model__max_features": [
        "sqrt",
        0.5
    ]
}

In [204]:
# 4.3 Randomized search

# We use RandomizedSearchCV to test different parameter combinations and find a better model.

random_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,
    param_distributions=param_distributions,

    # Test 4 parameter combinations
    n_iter=4,

    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },

    refit="MAE",

    # Use the same CV as before
    cv=cv_splits,

    random_state=42,

    # Keep resource usage reasonable
    n_jobs=1,

    return_train_score=False
)

In [205]:
# 4.4 Run search

# We run the search to find the best parameter combination.

print("Starting RandomizedSearchCV...")

random_search.fit(
    X_train,
    y_train
)

print("RandomizedSearchCV completed.")

Starting RandomizedSearchCV...
RandomizedSearchCV completed.


In [206]:
# 4.5 Best hyperparameters

# We check which parameter combination gave the best CV result.

print("\nBest hyperparameters:")

for parameter, value in random_search.best_params_.items():
    print(f"{parameter}: {value}")


Best hyperparameters:
model__n_estimators: 30
model__min_samples_split: 5
model__min_samples_leaf: 2
model__max_features: 0.5
model__max_depth: 16


In [207]:
# The best Random Forest configuration uses 30 trees, a maximum depth of 16, max_features=0.5, min_samples_split=5, and min_samples_leaf=2

In [208]:
# 4.6 Best CV MAE

# We check the MAE of the best tuned model from cross-validation.

best_cv_mae = -random_search.best_score_

print(
    "\nBest CV MAE:",
    f"{best_cv_mae:,.2f} AZN"
)


Best CV MAE: 110,468.21 AZN


In [209]:
# The best tuned Random Forest achieved a cross-validation MAE of 110,468.21 AZN

In [210]:
# 4.7 Best search results

# We check the CV performance of the best tuned model.

results = pd.DataFrame(
    random_search.cv_results_
)

best_index = random_search.best_index_

best_mae_mean = -results.loc[
    best_index, "mean_test_MAE"
]

best_mae_std = results.loc[
    best_index, "std_test_MAE"
]

best_rmse_mean = -results.loc[
    best_index, "mean_test_RMSE"
]

best_rmse_std = results.loc[
    best_index, "std_test_RMSE"
]

best_r2_mean = results.loc[
    best_index, "mean_test_R2"
]

best_r2_std = results.loc[
    best_index, "std_test_R2"
]

print("\nBest tuned model CV results:")

print(
    f"MAE: {best_mae_mean:,.2f} +/- {best_mae_std:,.2f}"
)

print(
    f"RMSE: {best_rmse_mean:,.2f} +/- {best_rmse_std:,.2f}"
)

print(
    f"R2: {best_r2_mean:.4f} +/- {best_r2_std:.4f}"
)


Best tuned model CV results:
MAE: 110,468.21 +/- 4,269.20
RMSE: 1,801,922.36 +/- 1,334,459.18
R2: -0.0652 +/- 0.0668


In [211]:
# The tuned Random Forest achieved MAE = 110,468.21 ± 4,269.20 AZN, RMSE = 1,801,922.36 ± 1,334,459.18 AZN, and R² = −0.0652 ± 0.0668 in cross-validation

In [212]:
# 4.8 Baseline vs tuned Random Forest

# We compare the tuned model with the original Random Forest to see if it improved.

baseline_row = comparison_df[
    comparison_df["Model"] == "Random Forest"
].iloc[0]

baseline_mae = baseline_row["MAE Mean"]

improvement = (
    (baseline_mae - best_mae_mean)
    / baseline_mae
    * 100
)

print(
    "\nBaseline Random Forest CV MAE:",
    f"{baseline_mae:,.2f}"
)

print(
    "Tuned Random Forest CV MAE:",
    f"{best_mae_mean:,.2f}"
)

print(
    "MAE improvement:",
    f"{improvement:.2f}%"
)


Baseline Random Forest CV MAE: 122,429.97
Tuned Random Forest CV MAE: 110,468.21
MAE improvement: 9.77%


In [213]:
# The tuned Random Forest reduced CV MAE from 122,429.97 AZN to 110,468.21 AZN, achieving a 9.77% improvement over the baseline.

In [214]:
# 4.9 Best model

# We keep the best tuned Random Forest to use for the final evaluation.

best_random_forest = (
    random_search.best_estimator_
)

print("\nBest tuned Random Forest:")
print(best_random_forest)


Best tuned Random Forest:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['area_m2', 'area_value',
                                                   'floor', 'total_floors',
                                                   'Otaq sayı', 'lat', 'lng',
                                                   'views']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Missing',
        

In [215]:
# The best tuned Random Forest pipeline was saved for final evaluation, using preprocessing for numerical and categorical features followed by the optimized Random Forest model

In [216]:
# 4.10 Search summary

# We review all tested parameter combinations and their CV results.

search_summary = results[
    [
        "params",
        "mean_test_MAE",
        "std_test_MAE",
        "mean_test_RMSE",
        "std_test_RMSE",
        "mean_test_R2",
        "std_test_R2"
    ]
].copy()

search_summary["mean_test_MAE"] = (
    -search_summary["mean_test_MAE"]
)

search_summary["mean_test_RMSE"] = (
    -search_summary["mean_test_RMSE"]
)

search_summary = (
    search_summary
    .sort_values("mean_test_MAE")
    .reset_index(drop=True)
)

search_summary.columns = [
    "Parameters",
    "MAE Mean",
    "MAE Std",
    "RMSE Mean",
    "RMSE Std",
    "R2 Mean",
    "R2 Std"
]

display(search_summary)

,Parameters,MAE Mean,MAE Std,RMSE Mean,RMSE Std,R2 Mean,R2 Std
0,"{'model__n_estimators': 30, 'model__min_sample...",110468.208569,4269.199092,1.801922e+06,1.334459e+06,-0.065172,0.066835
1,"{'model__n_estimators': 20, 'model__min_sample...",114845.690890,1928.517528,1.924439e+06,1.248855e+06,-0.416575,0.331745
2,"{'model__n_estimators': 30, 'model__min_sample...",121849.292678,5937.282549,1.841597e+06,1.307296e+06,-0.181016,0.265033
3,"{'model__n_estimators': 20, 'model__min_sample...",125360.955363,8485.379701,1.904317e+06,1.269049e+06,-0.371860,0.461816


In [217]:
# The hyperparameter search shows that the best configuration achieved the lowest CV MAE of 110,468.21 AZN, outperforming all other tested configurations.

In [218]:
# 4.11 Test set protection

# We make sure the test set was kept separate during hyperparameter tuning.

assert "X_test" in globals()
assert "y_test" in globals()

print(
    "\nTest set was not used during hyperparameter tuning."
)


Test set was not used during hyperparameter tuning.


In [219]:
# 4.12 Final checks

# We check that the tuning process finished correctly and produced valid results.

assert random_search.best_estimator_ is not None
assert random_search.best_params_ is not None

assert np.isfinite(best_mae_mean)
assert np.isfinite(best_rmse_mean)
assert np.isfinite(best_r2_mean)

assert len(results) == 4

assert (
    results["mean_test_MAE"]
    .notna()
    .all()
)


Checkpoint 4 Interpretation

In this checkpoint, we tuned the Random Forest model using RandomizedSearchCV while keeping the same group-based cross-validation as in Checkpoint 3. The test set was not used during tuning.

The tuned Random Forest performed better than the baseline model. Its MAE decreased from 122,429.97 AZN to 110,468.21 AZN, which means the model improved by about 9.77%. This shows that the selected hyperparameters helped the model make more accurate price predictions on average.

However, the R² value was -0.0652, which is still a weakness. It means that the model is not explaining the overall variation in house prices very well. The very high RMSE compared with the MAE also suggests that there are some properties where the prediction error is extremely large. This is likely related to the unusual and extreme house prices found during the EDA stage.

Overall, the tuning improved the Random Forest model, especially in terms of MAE, but the model still has limitations. The next step is to test this tuned model on the untouched test set to see how well it performs on completely unseen properties.

# Checkpoint 5 — Final Model Evaluation

In this step, I evaluated the tuned Random Forest on the separate test set.

The test set was not used during model comparison or hyperparameter tuning. It was used only at this final stage to get an honest estimate of how the model performs on unseen properties.

The final model was evaluated using **MAE, RMSE, and R²**. The final test results were:

* **MAE:** 94,649.45 AZN
* **RMSE:** 700,392.96 AZN
* **R²:** -0.1185

The model performed better in terms of MAE on the test set than its cross-validation result. However, the negative R² shows that the model still has difficulty explaining the full variation in house prices.

Overall, these results represent the final performance of the selected model on unseen data.


In [220]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [221]:
# 5.1 Final model

# We use the tuned Random Forest from Checkpoint 4 for the final test.

final_model = best_random_forest

print("Final model:")
print(final_model)

Final model:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['area_m2', 'area_value',
                                                   'floor', 'total_floors',
                                                   'Otaq sayı', 'lat', 'lng',
                                                   'views']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Missing',
                      

In [222]:
# The tuned Random Forest from Checkpoint 4 was selected as the final model for one-time evaluation on the unseen test set

In [223]:
# 5.2 Make predictions on the untouched test set

# We make predictions on the test set only once for the final evaluation.

print("\nEvaluating final model on the test set...")

y_test_pred = final_model.predict(X_test)

print("Test predictions completed.")


Evaluating final model on the test set...
Test predictions completed.


In [224]:
# 5.3 Calculate final test metrics

# We calculate the final MAE, RMSE, and R² on the untouched test set.

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

test_r2 = r2_score(
    y_test,
    y_test_pred
)

In [225]:
# 5.4 Display final results

# We display the final test results to understand how well the model performs on unseen data.

print("\n" + "=" * 70)
print("FINAL TEST SET RESULTS")
print("=" * 70)

print(f"MAE:  {test_mae:,.2f} AZN")
print(f"RMSE: {test_rmse:,.2f} AZN")
print(f"R²:   {test_r2:.4f}")


FINAL TEST SET RESULTS
MAE:  94,649.45 AZN
RMSE: 700,392.96 AZN
R²:   -0.1185


In [226]:
# The final model achieved a test MAE of 94,649.45 AZN, RMSE of 700,392.96 AZN, and R² of −0.1185 on unseen data.

In [227]:
# 5.5 Compare CV and test performance

# We compare CV results with the final test results to check how well the model generalizes.

print("\n" + "=" * 70)
print("CV VS TEST PERFORMANCE")
print("=" * 70)

print(
    f"Cross-validated MAE: "
    f"{best_mae_mean:,.2f} +/- {best_mae_std:,.2f} AZN"
)

print(
    f"Final test MAE: "
    f"{test_mae:,.2f} AZN"
)

print(
    f"Cross-validated RMSE: "
    f"{best_rmse_mean:,.2f} +/- {best_rmse_std:,.2f} AZN"
)

print(
    f"Final test RMSE: "
    f"{test_rmse:,.2f} AZN"
)

print(
    f"Cross-validated R²: "
    f"{best_r2_mean:.4f} +/- {best_r2_std:.4f}"
)

print(
    f"Final test R²: "
    f"{test_r2:.4f}"
)


CV VS TEST PERFORMANCE
Cross-validated MAE: 110,468.21 +/- 4,269.20 AZN
Final test MAE: 94,649.45 AZN
Cross-validated RMSE: 1,801,922.36 +/- 1,334,459.18 AZN
Final test RMSE: 700,392.96 AZN
Cross-validated R²: -0.0652 +/- 0.0668
Final test R²: -0.1185


In [228]:
# The test set shows lower MAE and RMSE than CV, but the R² is slightly worse, indicating limited generalization despite lower prediction errors

In [229]:
# 5.6 Test set prediction summary

# We compare actual and predicted prices to see the model's individual errors.

prediction_summary = pd.DataFrame(
    {
        "Actual Price": y_test.values,
        "Predicted Price": y_test_pred,
        "Absolute Error": np.abs(
            y_test.values - y_test_pred
        )
    }
)

print("\nPrediction examples:")

display(
    prediction_summary.head(10)
)


Prediction examples:


,Actual Price,Predicted Price,Absolute Error
0,365000.0,438356.986570,73356.986570
1,161000.0,179445.507889,18445.507889
2,155000.0,149906.722483,5093.277517
3,190000.0,214391.943016,24391.943016
4,178000.0,150928.600578,27071.399422
5,175000.0,158709.474149,16290.525851
6,230000.0,240845.182468,10845.182468
7,140000.0,135508.256397,4491.743603
8,137000.0,151164.183796,14164.183796
9,137000.0,153107.132311,16107.132311


In [230]:
# The prediction examples show that the model generally produces reasonable estimates

In [231]:
# 5.7 Final checks

# We check that the final predictions and evaluation metrics are valid.

assert len(y_test_pred) == len(y_test)

assert np.isfinite(test_mae)
assert np.isfinite(test_rmse)
assert np.isfinite(test_r2)

assert test_mae >= 0
assert test_rmse >= 0



In [232]:
# Final test results

# We display the final performance of the model on unseen data.

print(f"Final Test MAE:  {test_mae:,.2f} AZN")
print(f"Final Test RMSE: {test_rmse:,.2f} AZN")
print(f"Final Test R²:   {test_r2:.4f}")

Final Test MAE:  94,649.45 AZN
Final Test RMSE: 700,392.96 AZN
Final Test R²:   -0.1185


Checkpoint 5 Interpretation

The tuned Random Forest was evaluated on the separate test set for the first and only time at the end of the project. The model achieved a MAE of 94,649.45 AZN, meaning that, on average, its predicted house prices were about 94.6 thousand AZN away from the actual prices.

The RMSE was 700,392.96 AZN, which is much higher than the MAE. This suggests that although the model performs reasonably on many properties, there are some predictions with very large errors. The extreme house prices identified during the EDA stage may be contributing to these large errors.

The final R² was -0.1185, meaning that the model still does not explain the overall variation in house prices very well. However, the final test MAE of 94,649.45 AZN is lower than the cross-validation MAE of 110,468.21 AZN, so the model performed somewhat better on this particular unseen test set than expected from cross-validation.

Overall, the tuned Random Forest provides useful predictions for many properties, but there is still a significant amount of variation in house prices that the current features do not capture. Since the test set was kept completely separate until this final step, these results can be treated as the final evaluation of the model.

# Checkpoint 6 — Saving the Final Model

The final tuned Random Forest model was saved using `joblib` so it can be reused later without retraining the model.

After saving the model, I checked that the file was created successfully and loaded it again to make sure it could still generate predictions on the test data. The first five predictions were also checked, and the number of predictions was confirmed to match the test set size.

This ensures that the final model was saved correctly and can be loaded and used for future house price predictions.


In [235]:

import joblib
import os


# 6.1 Save the final tuned model

model_filename = "final_house_price_model.joblib"

joblib.dump(
    best_random_forest,
    model_filename
)

print(f"Final model saved as: {model_filename}")


# 6.2 Make sure the model file was created

assert os.path.exists(model_filename)

print(
    "Model file size:",
    os.path.getsize(model_filename),
    "bytes"
)


# 6.3 Load the saved model again

loaded_model = joblib.load(model_filename)



# 6.4 Check that the loaded model still works

loaded_predictions = loaded_model.predict(X_test)

print("\nFirst 5 predictions from the loaded model:")
print(loaded_predictions[:5])

assert len(loaded_predictions) == len(X_test)


Final model saved as: final_house_price_model.joblib
Model file size: 10906259 bytes

First 5 predictions from the loaded model:
[438356.98657013 179445.5078895  149906.72248292 214391.94301572
 150928.6005776 ]


# Checkpoint 7 — Non-Technical Report

## Project Overview

The goal of this project was to build a machine learning model that can predict house prices based on property characteristics such as location, area, number of rooms, floor, property condition, and other available information.

Several models were compared using the same group-aware cross-validation approach. Random Forest performed best based on Mean Absolute Error (MAE), so it was selected and further improved through hyperparameter tuning.

The tuned Random Forest achieved a cross-validation MAE of **110,468.21 AZN ± 4,269.20 AZN**. The final evaluation was then performed once on a completely separate test set. The model achieved a **test MAE of 94,649.45 AZN**, meaning that its predictions were off by about 94.6 thousand AZN on average.

The final test RMSE was **700,392.96 AZN**, while the R² score was **−0.1185**. The negative R² indicates that the model still has limitations in explaining the full variation in house prices, particularly because the dataset contains highly unusual prices and property sizes.

## Business Value

The model can provide a useful starting point for estimating property prices and identifying general price patterns in the housing market. It could potentially support agents, property platforms, or analysts by providing an automated price estimate based on available property information.

However, the predictions should not be treated as exact market prices. The model would benefit from cleaner data, more consistent property information, and additional features that better capture factors affecting real estate prices.

## Final Recommendation

The tuned Random Forest model is suitable as a baseline price prediction tool, but further improvements should be made before using it for important business decisions. Future work could focus on handling extreme values, adding more relevant property features, and testing additional modeling approaches.
